In [1]:
import pandas as pd
import numpy as np


In [2]:
data = pd.read_csv("../../data/MRegularSeasonCompactResults.csv")

In [3]:
data

In [4]:
def get_team_matches(team_id, years_range=None):
    if years_range:
        return data[((data.WTeamID == team_id) | (data.LTeamID == team_id)) & (data.Season.isin(years_range))]
    return data[(data.WTeamID == team_id) | (data.LTeamID == team_id)]

In [5]:
data.columns

In [6]:
get_team_matches(1101)

In [7]:
def calculate_elo_score_of_teams(data):
    elo = {}
    for index, row in data.iterrows():
        wteam = row.WTeamID
        lteam = row.LTeamID
        wteam_elo = elo.get(wteam, 1500)
        lteam_elo = elo.get(lteam, 1500)
        elo[wteam] = wteam_elo + 20 * (1 - 1 / (1 + 10 ** ((lteam_elo - wteam_elo) / 400)))
        elo[lteam] = lteam_elo + 20 * (0 - 1 / (1 + 10 ** ((wteam_elo - lteam_elo) / 400)))
    return elo

In [8]:
data.shape

In [9]:
def transform_team_data(team_id, games_df):
    team_games = games_df[(games_df['WTeamID'] == team_id) | (games_df['LTeamID'] == team_id)].copy()
    
    # Create new columns based on whether the team won or lost
    team_games['Opponent'] = np.where(team_games['WTeamID'] == team_id,
                                    team_games['LTeamID'],
                                    team_games['WTeamID'])
    
    team_games['win_loss'] = np.where(team_games['WTeamID'] == team_id, 1, 0)
    team_games['team'] = team_id
    
    # Add location logic
    team_games['Loc'] = np.where(
        team_games['WTeamID'] == team_id,
        np.where(team_games['WLoc'] == 'H', 1, 0),  # If team won
        np.where(team_games['WLoc'] == 'A', 1, 0)   # If team lost
    )

    margin_of_victory = team_games['WScore'] - team_games['LScore']
    team_games['margin_of_victory'] = np.where(team_games['WTeamID'] == team_id,
                                              margin_of_victory,
                                              -margin_of_victory)
    
    # Select and rename columns to match desired format
    result = team_games[[
        'Season',
        'DayNum',
        'team',
        'WScore',
        'LScore',
        'Loc',    
        'NumOT',
        'Opponent',
        'WLoc',
        'margin_of_victory',
        'win_loss'
    ]]
    
    result = result.sort_values(['Season', 'DayNum'])
    
    return result
    

In [10]:
team_id = 1411
team_history = transform_team_data(team_id, data)
team_history

In [11]:
def generate_team_data_for_all(matches_df):
    unique_teams = set(matches_df['WTeamID']).union(set(matches_df['LTeamID']))
    all_teams_df = pd.concat([transform_team_data(team_id, matches_df) for team_id in unique_teams], ignore_index=True)
    
    return all_teams_df

In [12]:
all_teams_data = generate_team_data_for_all(data)
all_teams_data

In [13]:
team_history.columns

In [14]:
elo_scores = calculate_elo_score_of_teams(data)

# Pagerank

In [15]:
pip install networkx

In [16]:
import networkx as nx

In [17]:
all_teams_data

In [18]:
# Initialize directed graph
G = nx.DiGraph()

# Add nodes (teams)
teams = pd.concat([all_teams_data['Opponent'], all_teams_data['WLoc']]).unique()  # adjust as needed for team names
G.add_nodes_from(teams)

# Add weighted edges from loser to winner
for idx, row in all_teams_data.iterrows():
    loser = row['Opponent']  # ensure this is the losing team identifier
    winner = row['WLoc']     # adjust based on your actual columns for team IDs
    weight = row['margin_of_victory']   # or some function of margin, adjust for OT/home advantage if desired
    if G.has_edge(loser, winner):
        G[loser][winner]['weight'] += weight
    else:
        G.add_edge(loser, winner, weight=weight)

# Compute PageRank with weights
pagerank_scores = nx.pagerank(G, weight='weight')

# Convert PageRank differences to win probabilities using a logistic function
def predict_win_prob(team_A, team_B, beta=10):  # beta can be tuned
    r_A = pagerank_scores.get(team_A, 0)
    r_B = pagerank_scores.get(team_B, 0)
    return 1 / (1 + np.exp(-beta * (r_A - r_B)))



In [19]:
# Example prediction for a matchup
team_A = '1001'
team_B = '1002'
predicted_prob = predict_win_prob(team_A, team_B)

print(f"Predicted probability that {team_A} wins over {team_B}: {predicted_prob:.2f}")